In [7]:
import pandas as pd
from pathlib import Path

snapshot_path = Path("/home/nixos/workspace/TradePilot/data/lakehouse/derived/derived.etf_aw_rebalance_snapshot/2026/04/part-00000.parquet")
snapshot = pd.read_parquet(snapshot_path)

snapshot[[
    "rebalance_date",
    "sleeve_code",
    "sleeve_role",
    "return_1m",
    "return_3m",
    "return_6m",
    "data_status",
]]

,rebalance_date,sleeve_code,sleeve_role,return_1m,return_3m,return_6m,data_status
0,2026-04-20,159001.SZ,cash,-0.000020,-0.000010,-0.000020,complete
1,2026-04-20,159845.SZ,equity_small,0.054841,-0.006062,0.110645,complete
2,2026-04-20,510300.SH,equity_large,0.036095,-0.004994,0.041872,complete
3,2026-04-20,511010.SH,bond,0.003763,0.010447,0.013153,complete
4,2026-04-20,518850.SH,gold,-0.009009,0.027724,0.135293,complete


In [8]:
from datetime import date
from pprint import pprint
from tradepilot.etl.service import ETLService

service = ETLService()
result = service.run_bootstrap(
    "derived.etf_aw_regime_score.build",
    start=date(2026, 4, 1),
    end=date(2026, 4, 30),
)
pprint(result)

{'dataset_name': 'derived.etf_aw_regime_score',
 'label_counts': {'mixed': 1},
 'partitions_written': 1,
 'profile_name': 'derived.etf_aw_regime_score.build',
 'records_inserted': 1,
 'records_updated': 0,
 'records_written': 1,
 'requested_end': '2026-04-30',
 'requested_start': '2026-04-01',
 'scoring_status_counts': {'complete': 1},
 'status': 'success',
 'storage_paths': ['derived/derived.etf_aw_regime_score/2026/04/part-00000.parquet'],
 'validation': {'confidence_cap_valid': True,
                'confidence_score_capped': True,
                'confidence_score_finite': True,
                'json_fields_valid': True,
                'label_allowed': True,
                'market_score_finite': True,
                'market_score_in_range': True,
                'no_budget_weight_trade_fields': True,
                'no_duplicate_business_keys': True,
                'non_empty': True,
                'schema_version_supported': True,
                'scoring_status_allowed': Tr

In [9]:
import json
import pandas as pd
from pathlib import Path

score_path = Path("/home/nixos/workspace/TradePilot/data/lakehouse/derived/derived.etf_aw_regime_score/2026/04/part-00000.parquet")
df = pd.read_parquet(score_path)

df[[
    "rebalance_date",
    "input_snapshot_status",
    "scoring_status",
    "market_regime_label",
    "market_score",
    "confidence_score",
    "confidence_cap",
]]

,rebalance_date,input_snapshot_status,scoring_status,market_regime_label,market_score,confidence_score,confidence_cap
0,2026-04-20,complete,complete,mixed,23.5,0.258,0.7


In [10]:
row = df.iloc[0]
signals = pd.DataFrame(json.loads(row["signals_json"]))

signals[[
    "sleeve_code",
    "sleeve_role",
    "direction_score",
    "data_status",
    "metric_signals",
]]

,sleeve_code,sleeve_role,direction_score,data_status,metric_signals
0,511010.SH,bond,0.0,complete,"{'return_1m': 0, 'return_3m': 0, 'return_6m': 0}"
1,159001.SZ,cash,0.0,complete,"{'return_1m': 0, 'return_3m': 0, 'return_6m': 0}"
2,510300.SH,equity_large,25.0,complete,"{'return_1m': 100, 'return_3m': 0, 'return_6m'..."
3,159845.SZ,equity_small,55.0,complete,"{'return_1m': 100, 'return_3m': 0, 'return_6m'..."
4,518850.SH,gold,30.0,complete,"{'return_1m': 0, 'return_3m': 0, 'return_6m': ..."


In [11]:
json.loads(row["quality_notes"])

{'agreement_score': 0.25,
 'confidence_cap_reasons': ['market_only_inputs'],
 'drawdown_penalty': 0.15,
 'input_data_status_counts': {'complete': 5},
 'macro_inputs_available': False,
 'market_only': True,
 'rates_inputs_available': False,
 'strength_score': 0.235,
 'volatility_penalty': 0.1}